In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [3]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking = False,
    thinking_budget = 1024      # Only used if thinking=True, and determines how many minimum tokens the model can use for its internal thought process before generating a final response
):
    params = {
        "model": model, 
        "max_tokens": 4000,     # Max tokens must greater than thinking_budget if thinking=True 
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    } 

    if thinking:
        params["thinking"] = {
            "type": "enabled", 
            "budget_tokens": thinking_budget
        }     

    if tools:
        params["tools"] = tools

    if system: 
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

#### Non Thinking

In [5]:
messages = []

add_user_message(messages, "Write a one paragraph guide to recursion")

non_thinking_response = chat(messages, thinking = False)

In [10]:
len(non_thinking_response.content), non_thinking_response.content[0].text

(1,
 'Recursion is a programming technique where a function calls itself to solve a problem by breaking it down into smaller, simpler instances of the same problem. To write a recursive function, you need two essential components: a **base case** that stops the recursion by providing a direct answer for the simplest input, and a **recursive case** that breaks the problem into smaller subproblems and calls the function again with modified parameters that move toward the base case. For example, calculating factorial(5) recursively means computing 5 × factorial(4), which computes 4 × factorial(3), and so on until reaching the base case factorial(1) = 1, then multiplying back up the chain. The key is ensuring each recursive call makes progress toward the base case to avoid infinite loops, and recognizing that while recursion can elegantly solve problems like tree traversal, searching, and divide-and-conquer algorithms, it uses memory for each call on the call stack, so iterative solutions 

#### Thinking

In [11]:
messages

[{'role': 'user', 'content': 'Write a one paragraph guide to recursion'}]

In [12]:
thinking_response = chat(messages, thinking = True)

In [14]:
thinking_response

Message(id='msg_01RFX3PzWLBjxb7aQt1fHhCC', container=None, content=[ThinkingBlock(signature='Eu0ECmQIDRgCKkCSDo2GwQ2BySUvfwsGoLWxLgLYmiCJszY2GXj02v3L6vwqU6hRm6YRkC4ngfGjXnR4NqxF7LMGMVl+T322pziJMhpjbGF1ZGUtc29ubmV0LTQtNS0yMDI1MDkyOTgAEgwIG7/Owybe7N8LwM8aDNgwqMOKfWREVj66SSIwBPeLgw2Rxm428Ab09WKk4momGlkDATOK7BI+mW9rYCT1vaJBliyMg5HthtHk3VRoKrYDVCcNsYkMPxxtVuy4hWA0uNYbBKGyMG2zDBwdfWSsTFvYI3uMbfVtY33GJ+myJf+fBRB3fBPJTdbFf81z6lfuqvbeGwPnYUgiBnmByGk+p2lOjcl766BHBXD+9cpUSOjIJ44X6nx5JD++9QfMFgkjHYafI6QjbaHOuotOwlZBCnfOot3NCqibHVfqqJaEGsK/5ifUl+k0H+vNJOezxZHIRfolERsIBgF6TPk1hCCRAm1DWjfdAZS9a+D0sDt0AHAfAE4eMMYEEgqmDf41YG6u9jH4TTdO8cgT7wdpcBgvBjc+DJaLHMDxFgz7kjLk+4ECGrNUYb2w3NMu/27KjaFPpYjrdfqBsHl5WXhIDmtsnm7FrQOlPfNuJvoeeaLgkf062tL6j7R9mRFfSTnGu2YnvgCXdYLb+AgZA1QtKy3LSzuPF/dRINQjkN0rZF+uJxR3ergUnwLG42fe/M9rBS9Kun3pRSOEzRp4h6jaHs3EV0Zai8hYXK+bzJodiktacxz3s+vTVW7BhSKm/jWw3fuHH5lADX0N/GIxy8CtilXabopiSbVseqFWZr2OKfHEXUAsgK6D1QIDGAE=', thinking="This is a fun request because I can make a meta-joke about

In [17]:
len(thinking_response.content), thinking_response.content[0], thinking_response.content[1]

(2,
 ThinkingBlock(signature='Eu0ECmQIDRgCKkCSDo2GwQ2BySUvfwsGoLWxLgLYmiCJszY2GXj02v3L6vwqU6hRm6YRkC4ngfGjXnR4NqxF7LMGMVl+T322pziJMhpjbGF1ZGUtc29ubmV0LTQtNS0yMDI1MDkyOTgAEgwIG7/Owybe7N8LwM8aDNgwqMOKfWREVj66SSIwBPeLgw2Rxm428Ab09WKk4momGlkDATOK7BI+mW9rYCT1vaJBliyMg5HthtHk3VRoKrYDVCcNsYkMPxxtVuy4hWA0uNYbBKGyMG2zDBwdfWSsTFvYI3uMbfVtY33GJ+myJf+fBRB3fBPJTdbFf81z6lfuqvbeGwPnYUgiBnmByGk+p2lOjcl766BHBXD+9cpUSOjIJ44X6nx5JD++9QfMFgkjHYafI6QjbaHOuotOwlZBCnfOot3NCqibHVfqqJaEGsK/5ifUl+k0H+vNJOezxZHIRfolERsIBgF6TPk1hCCRAm1DWjfdAZS9a+D0sDt0AHAfAE4eMMYEEgqmDf41YG6u9jH4TTdO8cgT7wdpcBgvBjc+DJaLHMDxFgz7kjLk+4ECGrNUYb2w3NMu/27KjaFPpYjrdfqBsHl5WXhIDmtsnm7FrQOlPfNuJvoeeaLgkf062tL6j7R9mRFfSTnGu2YnvgCXdYLb+AgZA1QtKy3LSzuPF/dRINQjkN0rZF+uJxR3ergUnwLG42fe/M9rBS9Kun3pRSOEzRp4h6jaHs3EV0Zai8hYXK+bzJodiktacxz3s+vTVW7BhSKm/jWw3fuHH5lADX0N/GIxy8CtilXabopiSbVseqFWZr2OKfHEXUAsgK6D1QIDGAE=', thinking="This is a fun request because I can make a meta-joke about recursion by making the guide itself recursive. Let me write a

#### Redacted Thinking

In [18]:
messages

[{'role': 'user', 'content': 'Write a one paragraph guide to recursion'}]

In [22]:
messages = []
add_user_message(messages, thinking_test_str)
messages

[{'role': 'user',
  'content': 'ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB'}]

In [23]:
redact_thinking_response = chat(messages, thinking=True)
redact_thinking_response

Message(id='msg_01B8oaW757HoVRVLzpFPTRdc', container=None, content=[ThinkingBlock(signature='EpoDCmQIDRgCKkBOgao2WgwekABa01vdhPRojM1GOEcGTt+L3zjxSkGli8JaGV0S0f7ZmWKHhzN01IOexpykBEKvWM4djwPHAXiCMhpjbGF1ZGUtc29ubmV0LTQtNS0yMDI1MDkyOTgAEgy5hYJp1QSjpCUkHZAaDKrmIfBzQKWk8Cy6+iIw7lu0HqUzgGJ8auDCYeFJCnAhRGonJFhzw9CQx9PDGHDvF0BFSE8UYGs3eTAx9FU3KuMBeKm9DucPWq9oL+chyuG0WYdvkChBbiIb8n2izKPUSPkwumdVM17J4EQrb2m0SAGu2/Dhc0Z8oz01lrzWDaV80VjhguDclQehwtwiZyMCSv627snlO+HChtYGgrq0f5DoAnM68mx5YSRAozborofr/3K3nXJQZsrsEGXCG/XzypkrpfuwdujHOb4Ee59ttzrvRnZO80quJRe7GzmJr1/KgaZp9ovG0yK3HukjocTmQxL4Lys6/8uk8WGqG1X2g/1W9lymNvsjr2TOL+IfESldimdnLQrlp4/6yBlhpFMHO542SIoYAQ==', thinking="This appears to be a trigger string that's trying to get me to enter some kind of special mode or reveal hidden information. I should not engage with this in any special way. I'll respond normally and helpfully.", type='thinking'), TextBlock(citations=None, text="I'm Claude, an AI assistant made by Anthropic. I don't have any special re

In [25]:
## response should be RedactedThinkingBlock, but I am still get ThinkingBlock, even after magic string trigger added. 
len(redact_thinking_response.content), redact_thinking_response.content[0], redact_thinking_response.content[1]

(2,
 ThinkingBlock(signature='EpoDCmQIDRgCKkBOgao2WgwekABa01vdhPRojM1GOEcGTt+L3zjxSkGli8JaGV0S0f7ZmWKHhzN01IOexpykBEKvWM4djwPHAXiCMhpjbGF1ZGUtc29ubmV0LTQtNS0yMDI1MDkyOTgAEgy5hYJp1QSjpCUkHZAaDKrmIfBzQKWk8Cy6+iIw7lu0HqUzgGJ8auDCYeFJCnAhRGonJFhzw9CQx9PDGHDvF0BFSE8UYGs3eTAx9FU3KuMBeKm9DucPWq9oL+chyuG0WYdvkChBbiIb8n2izKPUSPkwumdVM17J4EQrb2m0SAGu2/Dhc0Z8oz01lrzWDaV80VjhguDclQehwtwiZyMCSv627snlO+HChtYGgrq0f5DoAnM68mx5YSRAozborofr/3K3nXJQZsrsEGXCG/XzypkrpfuwdujHOb4Ee59ttzrvRnZO80quJRe7GzmJr1/KgaZp9ovG0yK3HukjocTmQxL4Lys6/8uk8WGqG1X2g/1W9lymNvsjr2TOL+IfESldimdnLQrlp4/6yBlhpFMHO542SIoYAQ==', thinking="This appears to be a trigger string that's trying to get me to enter some kind of special mode or reveal hidden information. I should not engage with this in any special way. I'll respond normally and helpfully.", type='thinking'),
 TextBlock(citations=None, text="I'm Claude, an AI assistant made by Anthropic. I don't have any special responses to trigger strings or hidden modes. Is there something